In [2]:
# Librerías
import pandas as pd
from pathlib import Path
import ee

/home/ddayann/miniconda3/envs/tf_cuda_gee/lib/python3.10/site-packages/google/api_core/_python_version_support.py:273: FutureWarning: You are using a Python version (3.10.19) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


#### Mecanismo de autorización
1. Al ejecutar las siguientes líneas se habilitará un link de acceso
2. Asegura los permisos apropiados para google Earth Engine y demás necesarios
3. Se genera un Token
4. Copiar y pegar el token en la ventana emergente que aparece en la ventana superior de VSC

* Para habilitar nuevos usuarios es necesario desmarcar "ee.Authenticate(force=True)" - Actualmente ya tiene habilitado el ingreso del usuario presente.  
(4/1AeoWuM9_9T1gCVAvTELP8S2YXcFW26H8_p7npV3xK4wc_OUw6Fb5i68ThS8)

In [2]:
# Log Google Earth Engine
# ee.Authenticate(force=True)
ee.Initialize(project='ee-cafe-494102')

In [3]:
# Cargar asset de municipios
municipios_ee = ee.FeatureCollection("projects/ee-cafe-494102/assets/municipios")

# Validación
print("Número de municipios:", municipios_ee.size().getInfo())
print("Columnas:", municipios_ee.first().propertyNames().getInfo())

Número de municipios: 27
Columnas: ['OBJECTID', 'MpNombre', 'Depto', 'MpCodigo', 'MpCategor', 'SHAPE_Leng', 'MpAltitud', 'SHAPE_Area', 'MpArea', 'system:index', 'MpNorma', 'Restriccio']


In [4]:
# Fechas de análisis
fecha_inicio = "2005-01-01"
fecha_fin = "2026-01-01"

In [6]:
# Cargar CHIRPS diario
chirps = (
    ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY")
    .filterDate(fecha_inicio, fecha_fin)
    .select("precipitation")
)

In [9]:
# Reducir cada imagen diaria por municipio
def reducir_precipitacion_diaria(img):
    fecha = ee.Date(img.get("system:time_start")).format("YYYY-MM-dd")

    reduccion = img.reduceRegions(
        collection=municipios_ee,
        reducer=ee.Reducer.mean()
            .combine(
                reducer2=ee.Reducer.count(),
                sharedInputs=True
            ),
        scale=5566
    )

    def limpiar_feature(feat):
        return ee.Feature(None, {
            "municipio": feat.get("MpNombre"),
            "date": fecha,
            "precip_mm": feat.get("mean"),
            "n_pixeles": feat.get("count")
        })

    return reduccion.map(limpiar_feature)

In [10]:
# Aplicar reducción diaria y aplanar resultados
precipitacion_municipal = chirps.map(reducir_precipitacion_diaria).flatten()

In [11]:
# Exportar a Google Drive sin geometría
task = ee.batch.Export.table.toDrive(
    collection=precipitacion_municipal,
    description="prcipitacion_municipal_chirps_2005_2025",
    folder="GEE_exports",
    fileNamePrefix="precipitacion_municipal_chirps_2005_2025",
    fileFormat="CSV",
    selectors=[
        "municipio",
        "date",
        "precip_mm",
        "n_pixeles"
    ]
)
task.start()

print("Exportación iniciada.")
print("Revisa el estado de la tarea en Google Earth Engine Tasks.")

Exportación iniciada.
Revisa el estado de la tarea en Google Earth Engine Tasks.


#### Validación del proceso 
A fin de validar que el proceso se está ejecutando es necesario abrir el proyecto de Google Earth Engine, en la pestaña Task. Es posible ver el avance del proceso.  
Al final del procesamiento el archivo será guardado en Google Drive: "/MyDrive/Gee_exports"

In [3]:
# Lectura archivo CSV exportado
BASE_DIR = Path.cwd().parents[1]
ruta_precipitacion = BASE_DIR / "data" / "raw" / "precipitacion_municipal_chirps_2005_2025.csv"
df_precipitacion = pd.read_csv(ruta_precipitacion)

df_precipitacion

,municipio,date,precip_mm,n_pixeles
0,Villamaria,2005-01-01,0.000000,25
1,Risaralda,2005-01-01,0.000000,9
2,Neira,2005-01-01,0.000000,23
3,Anserma,2005-01-01,0.000000,16
4,Manzanares,2005-01-01,0.000000,12
...,...,...,...,...
207085,Belalcázar,2025-12-31,0.000000,13
207086,San José,2025-12-31,0.000000,7
207087,Marquetalia,2025-12-31,0.000000,8
207088,Norcasia,2025-12-31,0.000000,16


In [4]:
df_precipitacion.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 207090 entries, 0 to 207089
Data columns (total 4 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   municipio  207090 non-null  object 
 1   date       207090 non-null  object 
 2   precip_mm  207090 non-null  float64
 3   n_pixeles  207090 non-null  int64  
dtypes: float64(1), int64(1), object(2)
memory usage: 6.3+ MB


In [5]:
# Tipificación
df_precipitacion["municipio"] = df_precipitacion["municipio"].astype("category")
df_precipitacion["date"] = pd.to_datetime(df_precipitacion["date"])


In [19]:
df_precipitacion.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 207090 entries, 0 to 207089
Data columns (total 4 columns):
 #   Column     Non-Null Count   Dtype         
---  ------     --------------   -----         
 0   municipio  207090 non-null  category      
 1   date       207090 non-null  datetime64[ns]
 2   precip_mm  207090 non-null  float64       
 3   n_pixeles  207090 non-null  int64         
dtypes: category(1), datetime64[ns](1), float64(1), int64(1)
memory usage: 4.9 MB


In [6]:
# Agregación mensual
df_precipitacion["mes"] = df_precipitacion["date"].dt.month
df_precipitacion["anio"] = df_precipitacion["date"].dt.year

# Agregación mensual
precipitacion_mes = (
    df_precipitacion
    .groupby(["municipio", "anio", "mes"], observed=False)
    .agg(
        precip_mm=("precip_mm", "sum"),          # acumulado mensual
        precip_mean_mm=("precip_mm", "mean"),    # precipitación diaria promedio del mes
        precip_max_mm=("precip_mm", "max"),      # mayor precipitación diaria del mes
        n_pixeles=("n_pixeles", "mean")
    )
    .reset_index()
)

# Crear fecha base: primer día del mes
precipitacion_mes["date"] = pd.to_datetime(
    precipitacion_mes["anio"].astype(str) + "-" +
    precipitacion_mes["mes"].astype(str) + "-01"
)

# Llevar al último día del mes
precipitacion_mes["date"] = (
    precipitacion_mes["date"] + pd.offsets.MonthEnd(0)
)

# Ordenar columnas
precipitacion_mes = precipitacion_mes[
    [
        "municipio",
        "date",
        "anio",
        "mes",
        "precip_mm",
        "precip_mean_mm",
        "precip_max_mm",
        "n_pixeles"
    ]
]

In [7]:
precipitacion_anual = (
    precipitacion_mes
    .groupby(["municipio", "anio"], observed=False)
    .agg(
        precip_mm=("precip_mm", "sum"),            # acumulado anual
        precip_mean_mm=("precip_mean_mm", "mean"), # promedio de los promedios mensuales
        precip_max_mm=("precip_max_mm", "max"),    # máximo mensual del año
        n_pixeles=("n_pixeles", "mean")
    )
    .reset_index()
)
# Fecha a cierre del año
precipitacion_anual["date"] = pd.to_datetime(
    precipitacion_anual["anio"].astype(str) + "-12-31"
)
precipitacion_anual["mes"] = 12
precipitacion_anual = precipitacion_anual[
    [
        "municipio",
        "date",
        "anio",
        "precip_mm",
        "precip_mean_mm",
        "precip_max_mm",
        "n_pixeles"
    ]
]

In [8]:
display(precipitacion_mes.head(12))
display(precipitacion_anual.head(21))

,municipio,date,anio,mes,precip_mm,precip_mean_mm,precip_max_mm,n_pixeles
0,Aguadas,2005-01-31,2005,1,114.152754,3.682347,30.293657,27.0
1,Aguadas,2005-02-28,2005,2,92.367963,3.298856,34.730915,27.0
2,Aguadas,2005-03-31,2005,3,212.648689,6.859635,30.772394,27.0
3,Aguadas,2005-04-30,2005,4,222.890335,7.429678,19.728767,27.0
4,Aguadas,2005-05-31,2005,5,365.459124,11.789004,38.967903,27.0
5,Aguadas,2005-06-30,2005,6,181.103017,6.036767,16.811892,27.0
6,Aguadas,2005-07-31,2005,7,117.711549,3.797147,19.614353,27.0
7,Aguadas,2005-08-31,2005,8,131.599857,4.245157,17.064086,27.0
8,Aguadas,2005-09-30,2005,9,179.689830,5.989661,29.864860,27.0
9,Aguadas,2005-10-31,2005,10,325.010335,10.484204,37.456772,27.0


,municipio,date,anio,precip_mm,precip_mean_mm,precip_max_mm,n_pixeles
0,Aguadas,2005-12-31,2005,2381.398658,6.503702,46.495879,27.0
1,Aguadas,2006-12-31,2006,2531.594393,6.926494,38.747605,27.0
2,Aguadas,2007-12-31,2007,2660.352575,7.242478,70.681276,27.0
3,Aguadas,2008-12-31,2008,3191.778940,8.714073,53.196217,27.0
4,Aguadas,2009-12-31,2009,2173.098247,5.956387,68.452368,27.0
5,Aguadas,2010-12-31,2010,3076.801927,8.415973,60.007248,27.0
6,Aguadas,2011-12-31,2011,2939.584554,8.061823,54.766049,27.0
7,Aguadas,2012-12-31,2012,2287.931082,6.247607,38.674836,27.0
8,Aguadas,2013-12-31,2013,2331.697350,6.393652,41.187441,27.0
9,Aguadas,2014-12-31,2014,1864.928067,5.111507,37.056813,27.0


In [9]:
# Exportar archivos finales
ruta_salida_mes = BASE_DIR / "data" / "processed" / "precipitacion_caldas_mes_2005-2025.csv"
precipitacion_mes.to_csv(ruta_salida_mes, index=False)
ruta_salida_ano = BASE_DIR / "data" / "processed" / "precipitacion_caldas_anio_2005-2025.csv"
precipitacion_anual.to_csv(ruta_salida_ano, index=False)